In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import numpy as np
import numpy.linalg as LA
import pandas as pd
import jax
import jax.numpy as jnp
from jax.typing import ArrayLike
import numpyro
import numpyro.distributions as dist

jax.config.update("jax_enable_x64", True)

import altair as alt

from principled_pclm.sim import PoissonEnveloped1D, make_aggregation_matrix
from principled_pclm.models._inference import run_inference_mcmc

from wavelet1d_utils import (
    make_haar_basis,
    get_basis_levels,
    make_wavelet_precision,
    wavelet_covariance_cholesky,
)
from principled_pclm.models import (
    expand_interval_dataframe,
    compute_interval_widths,
    build_composition_matrix,
)

/Users/shozen/Imperial/0_Research/mixres/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Generation

In [4]:
# Generate the dataset
left_limits = [0, 0, 5, 8, 10, 15, 18, 20, 25, 30, 35, 40, 50, 60, 70, 80, 90]
right_limits = [5, 12, 15, 25, 20, 30, 35, 40, 45, 50, 55, 60, 70, 80, 90, 95, 99]

# left_limits = [0, 10, 20, 30, 50, 70, 80, 90]
# right_limits = [9, 19, 29, 49, 69, 79, 89, 99]

intervals = [
    f"[{left},{right}{']' if right in {5, 15, 25, 30, 40, 50, 60, 70, 80, 90, 99} else ')'}"
    for left, right in zip(left_limits, right_limits)
]
print("Intervals:", intervals)

df_obs, df_rates = PoissonEnveloped1D(
    obs_intervals=intervals, n_observations=10, seed=42
).generate(return_true_rates=True)

# Expanded interval dataframe for plotting
df_expanded = expand_interval_dataframe(df_obs, interval_col="obs_interval", x_col="x")

Intervals: ['[0,5]', '[0,12)', '[5,15]', '[8,25]', '[10,20)', '[15,30]', '[18,35)', '[20,40]', '[25,45)', '[30,50]', '[35,55)', '[40,60]', '[50,70]', '[60,80]', '[70,90]', '[80,95)', '[90,99]']


In [5]:
def summarize_interval_segments(df_segment: pd.DataFrame) -> pd.DataFrame:
    return (
        df_segment[["obs_interval", "x", "lambda"]]
        .drop_duplicates(["obs_interval", "x"])
        .groupby("obs_interval", as_index=False)
        .agg(x_min=("x", "min"), x_max=("x", "max"), lambda_obs=("lambda", "mean"))
    )


def plot_data(df_true: pd.DataFrame, df_segment: pd.DataFrame) -> alt.Chart:
    source_segment = summarize_interval_segments(df_segment)

    chart_true = (
        alt.Chart(df_true)
        .mark_line(color="red")
        .encode(x="x", y=alt.Y("lambda:Q", title="lambda"))
    )

    chart_segment = (
        alt.Chart(source_segment)
        .mark_rule(color="orange", strokeWidth=2.5, opacity=0.85)
        .encode(
            x=alt.X("x_min:Q"),
            x2="x_max:Q",
            y=alt.Y("lambda_obs:Q", title="lambda"),
        )
    )

    return chart_true + chart_segment

In [6]:
plot_data(df_rates, df_expanded)

alt.LayerChart(...)

In [7]:
W = make_aggregation_matrix(left_limits, right_limits)
Phi = make_haar_basis(x=np.linspace(0, 100, 100), n_levels=5, include_scaling=False)
levels = get_basis_levels(5, include_scaling=False)
Q = make_wavelet_precision(levels, alpha=1.0, tau=1.0, jitter=1e-4)
K = wavelet_covariance_cholesky(Q)

## Result Plotting Helpers

Shared helpers for the 4 downstream experiments' result plots, so each experiment's own plot and the final side-by-side comparison are built from the same code.

In [8]:
def summarize_posterior_lam(
    mcmc_samples: dict, x: np.ndarray, label: str
) -> pd.DataFrame:
    """Posterior mean and 95% band for `lam`, tagged with an experiment label."""
    central = mcmc_samples["lam"].mean(axis=0)
    lower = np.percentile(mcmc_samples["lam"], 2.5, axis=0)
    upper = np.percentile(mcmc_samples["lam"], 97.5, axis=0)
    return pd.DataFrame(
        {
            "x": x,
            "lam": central.flatten(),
            "lower": lower.flatten(),
            "upper": upper.flatten(),
            "experiment": label,
        }
    )


def plot_posterior_band(
    source: pd.DataFrame,
    df_true: pd.DataFrame,
    df_segment: pd.DataFrame,
    title: str | None = None,
) -> alt.Chart:
    """Posterior mean + 95% band, layered over the true curve and raw interval markers."""
    chart = alt.Chart(source).mark_line(color="purple").encode(x="x", y="lam")
    band = (
        alt.Chart(source)
        .mark_area(color="purple", opacity=0.2)
        .encode(x="x", y="lower", y2="upper")
    )
    result = plot_data(df_true, df_segment) + chart + band
    return result.properties(title=title) if title else result

### Experiment 1: Naive Haar Wavelet model

In [9]:
def model_haar(
    W: ArrayLike,
    Phi: ArrayLike,
    N: ArrayLike,
    interval_id: ArrayLike,
    interval_width: ArrayLike,
    K: ArrayLike | None = None,
    y=None,
):
    """
    Naive Bayesian Haar wavelet regression model.

    Parameters
    ----------
    W : ArrayLike
        Aggregation matrix mapping fine-scale rates to observed intervals.
    Phi: ArrayLike
        Haar wavelet basis matrix.
    N: ArrayLike
        Number of observations in each interval.
    interval_id: ArrayLike
        Interval indices for each observation.
    interval_width: ArrayLike
        Width of each interval.
    K: ArrayLike | None, optional
        Cholesky factor of the coefficient covariance prior (Sigma = K @ K.T).
    y: ArrayLike, optional
        Observed counts in each interval. If None, the model will generate synthetic data.
    """
    M = Phi.shape[1]

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    beta = numpyro.sample("w", dist.Normal(0, 1).expand([M]).to_event(1))
    Phi_eff = Phi @ K if K is not None else Phi

    # --- Latent rate ---
    lam = numpyro.deterministic("lam", jnp.exp(beta0 + Phi_eff @ beta))
    lam_interval = (
        jnp.matmul(W, lam) / interval_width * N[interval_id]
    )  # Average rate over each interval (divide by interval-specific width)

    # --- Likelihood ---
    with numpyro.plate("obs", len(y)):
        numpyro.sample("y", dist.Poisson(lam_interval[interval_id]), obs=y)

In [10]:
df_model = df_obs.groupby(["obs_interval_id"])["y"].agg(N="size", y="sum").reset_index()

x = np.arange(0, 101)
y = df_model["y"].values
interval_id = df_model["obs_interval_id"].values
N = df_model["N"].values

intervals = df_obs["obs_interval"].unique().tolist()
A = build_composition_matrix(intervals)
interval_width = compute_interval_widths(intervals)

# Convert to JAX arrays
y_jax = jnp.array(y)
interval_id = jnp.array(interval_id)
N = jnp.array(N)
W_jax = jnp.array(W)
Phi_jax = jnp.array(Phi)
K_jax = jnp.array(K)
interval_width_jax = jnp.array(interval_width)

In [11]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(0)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_haar,
    W=W_jax,
    Phi=Phi_jax,
    N=N,
    interval_id=interval_id,
    interval_width=interval_width_jax,
    K=K_jax,
    y=y_jax,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:00<00:00, 4751.81it/s, 31 steps of size 2.32e-01. acc. prob=0.94]


Number of divergences: 0
dict_keys(['beta0', 'lam', 'w'])


In [12]:
source_exp1 = summarize_posterior_lam(
    mcmc_samples, np.linspace(0, 100, 100), "1. Naive"
)
plot_posterior_band(source_exp1, df_rates, df_expanded)

alt.LayerChart(...)